# MCP — Model Context Protocol

**MCP** (Model Context Protocol) — открытый стандарт для подключения LLM к внешним данным и инструментам.  
Вместо написания кастомного `@tool` для каждого источника — универсальный протокол:

```
LLM Agent  ←──MCP──→  MCP Server (БД, API, файлы...)
```

## Ключевые концепции

| Концепция | Описание |
|---|---|
| **Server** | Отдельный процесс, предоставляющий инструменты и ресурсы |
| **Tools** | Функции, которые LLM может вызвать (аналог `@tool` в LangChain) |
| **Resources** | Данные, которые LLM может прочитать: файлы, записи БД |
| **Client** | Приложение, которое подключается к серверу и передаёт инструменты агенту |

## Почему MCP, а не просто `@tool`?

- **Переносимость** — один сервер работает с любым клиентом (Claude Desktop, VS Code, ваш агент)
- **Изоляция** — сервер как отдельный процесс: можно обновлять и перезапускать независимо
- **Экосистема** — готовые серверы для GitHub, PostgreSQL, Slack, SQLite и других

In [ ]:
%pip install -q mcp langchain-mcp-adapters langchain-groq python-dotenv

## 1. MCP-сервер

Создаём простой сервер — хранилище заметок.  
LLM сможет читать и записывать заметки через MCP-протокол.

Сервер запускается как **отдельный процесс** — общение через `stdio`.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Notes Store")

# In-memory хранилище (в реальном проекте — БД)
_notes: dict[str, str] = {}  # key -> content


@mcp.tool()
def save_note(key: str, content: str) -> str:
    """Сохраняет заметку по ключу."""
    _notes[key] = content
    return f"Заметка \'{key}\' сохранена."


@mcp.tool()
def get_note(key: str) -> str:
    """Возвращает заметку по ключу."""
    return _notes.get(key, f"Заметка \'{key}\' не найдена.")


@mcp.tool()
def list_notes() -> list[str]:
    """Возвращает список всех ключей сохранённых заметок."""
    return list(_notes.keys()) if _notes else ["(заметок нет)"]


@mcp.tool()
def delete_note(key: str) -> str:
    """Удаляет заметку по ключу."""
    if key in _notes:
        del _notes[key]
        return f"Заметка \'{key}\' удалена."
    return f"Заметка \'{key}\' не найдена."


if __name__ == "__main__":
    mcp.run(transport="stdio")



## 2. Подключение клиента

`MultiServerMCPClient` запускает сервер как subprocess и **автоматически оборачивает**  
его инструменты в стандартные LangChain Tools — код агента не меняется.

In [ ]:
import sys
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

llm = ChatGroq(model="llama3-8b-8192", temperature=0)


async def ask_agent(question: str) -> str:
    """Запускает агента с MCP-инструментами."""
    async with MultiServerMCPClient(
        {
            "notes": {
                "command": sys.executable,
                "args": ["notes_server.py"],
                "transport": "stdio",
            }
        }
    ) as client:
        tools = client.get_tools()
        print(f"MCP-инструменты: {[t.name for t in tools]}")

        prompt = ChatPromptTemplate.from_messages([
            ("system", "Ты — ассистент для управления заметками. Используй инструменты для сохранения и чтения данных."),
            ("human", "{input}"),
            MessagesPlaceholder("agent_scratchpad"),
        ])

        agent = create_tool_calling_agent(llm, tools, prompt)
        executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

        result = await executor.ainvoke({"input": question})
        return result["output"]


print("Агент готов")

## 3. Демо

In [ ]:
# Сохраняем заметки через агента
result = await ask_agent(
    "Сохрани три заметки: "
    "'встреча' — 'встреча с командой в пятницу в 14:00', "
    "'задача' — 'доделать задание 4 до конца недели', "
    "'идея' — 'добавить кэширование в агента'"
)
print(result)

In [ ]:
# Читаем и суммаризируем сохранённые данные
result = await ask_agent("Покажи список всех заметок, потом прочитай каждую и кратко резюмируй")
print(result)

In [ ]:
# Удаляем заметку
result = await ask_agent("Удали заметку 'идея' и подтверди удаление")
print(result)

## 4. Архитектура

```
┌──────────────────────────────────────────────────┐
│                  Ваше приложение                 │
│                                                  │
│  ┌─────────────────┐  MCP   ┌─────────────────┐ │
│  │   LLM Agent     │ ←────→ │   MCP Server    │ │
│  │  (LangChain)    │ stdio  │                 │ │
│  └─────────────────┘        │  Tools:         │ │
│                             │  - save_note    │ │
│                             │  - get_note     │ │
│                             │  - list_notes   │ │
│                             └─────────────────┘ │
└──────────────────────────────────────────────────┘
```

## Транспорты MCP

| Транспорт | Использование |
|---|---|
| `stdio` | Локальный subprocess — учебный пример |
| `SSE` | HTTP-сервер, несколько клиентов |
| `WebSocket` | Двусторонняя связь в реальном времени |

## Готовые MCP-серверы

Агент подключает любой сервер — только добавить запись в конфиг `MultiServerMCPClient`:

```python
{
    "github":   {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-github"]},
    "postgres": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-postgres", DB_URL]},
    "sqlite":   {"command": "uvx", "args": ["mcp-server-sqlite", "--db-path", "data.db"]},
}
```